In [ ]:
rega_root <- Sys.getenv("REGA_SEPARATE_ROOT")
if (!nzchar(rega_root)) stop("Set REGA_SEPARATE_ROOT")

In [ ]:
library(ggplot2)

In [ ]:
heatmap_df<-read.table(file.path(rega_root, "02_run_rega/pathology_association/figures/selected_modules_heatmap_plot_data.tsv"),sep = "\t",header=TRUE)

In [ ]:
# Tile fill represents brain region; tile outline shows the sign of the
# linear pathology slope. Selected modules in the same cell share one
# region/cell-type slot and are drawn side by side.
region_colors <- c(
  "AG"  = "#8DA0CB",
  "EC"  = "#FC8D62",
  "HC"  = "#66C2A5",
  "MTC" = "#E78AC3",
  "PFC" = "#B3B3B3",
  "TH"  = "#A6D854"
)
celltype_order <- c("Exc", "Inh", "Ast", "Oli", "OPC", "Mic_Immune")
association_colors <- c("Positive" = "#B22222", "Negative" = "#2166AC")

pathology_df <- read.delim(
  file.path(rega_root, "02_run_rega/pathology_association/results/All_four_tests_H_pathology_results.tsv"),
  stringsAsFactors = FALSE
)
linear <- pathology_df[pathology_df$test == "Linear_Regression",
                       c("celltype", "region", "Module", "slope")]
selected_key <- paste(heatmap_df$celltype, heatmap_df$region,
                      heatmap_df$module_label, sep = "|")
linear_key <- paste(linear$celltype, linear$region, linear$Module, sep = "|")
if (anyDuplicated(selected_key) || anyDuplicated(linear_key)) {
  stop("Duplicate selected-module or linear-result keys")
}
matched <- match(selected_key, linear_key)
if (anyNA(matched)) {
  stop("Selected modules are missing from the linear pathology results")
}
heatmap_df$slope <- as.numeric(linear$slope[matched])
if (any(!is.finite(heatmap_df$slope)) || any(heatmap_df$slope == 0)) {
  stop("Every selected module needs a nonzero finite pathology slope")
}
heatmap_df$association <- ifelse(heatmap_df$slope > 0, "Positive", "Negative")

if (!all(grepl("^M[0-9]+$", heatmap_df$module_label))) {
  stop("Module labels must have the form M<number>")
}
heatmap_df$module_number <- as.integer(sub("^M", "", heatmap_df$module_label))
heatmap_df$region <- factor(heatmap_df$region, levels = names(region_colors))
heatmap_df$celltype <- factor(heatmap_df$celltype, levels = celltype_order)
if (anyNA(heatmap_df$region) || anyNA(heatmap_df$celltype)) {
  stop("Unexpected region or cell type in selected modules")
}
heatmap_df <- heatmap_df[
  order(heatmap_df$celltype, heatmap_df$region, heatmap_df$module_number),
]
pair <- interaction(heatmap_df$celltype, heatmap_df$region, drop = TRUE)
heatmap_df$modules_in_cell <- ave(
  rep(1L, nrow(heatmap_df)), pair, FUN = length
)
heatmap_df$module_position <- ave(
  seq_len(nrow(heatmap_df)), pair, FUN = seq_along
)
gap <- 0.02
heatmap_df$tile_width <- (0.88 - gap * (heatmap_df$modules_in_cell - 1)) /
  heatmap_df$modules_in_cell
heatmap_df$x <- as.integer(heatmap_df$region) +
  (heatmap_df$module_position - (heatmap_df$modules_in_cell + 1) / 2) *
  (heatmap_df$tile_width + gap)
heatmap_df$y <- as.integer(heatmap_df$celltype)
heatmap_df$xmin <- heatmap_df$x - heatmap_df$tile_width / 2
heatmap_df$xmax <- heatmap_df$x + heatmap_df$tile_width / 2
heatmap_df$ymin <- heatmap_df$y - 0.42
heatmap_df$ymax <- heatmap_df$y + 0.42

p <- ggplot(heatmap_df) +
  geom_rect(
    aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax,
        fill = region, color = association),
    linewidth = 1.4
  ) +
  geom_text(
    aes(x = x, y = y, label = module_label),
    color = "black", size = 4.2, fontface = "bold"
  ) +
  scale_fill_manual(values = region_colors, name = "Region") +
  scale_color_manual(
    values = association_colors,
    breaks = c("Positive", "Negative"),
    drop = FALSE,
    name = "Pathology association"
  ) +
  scale_x_continuous(
    breaks = seq_along(region_colors),
    labels = names(region_colors),
    limits = c(0.5, length(region_colors) + 0.5),
    expand = c(0, 0)
  ) +
  scale_y_continuous(
    breaks = seq_along(celltype_order),
    labels = celltype_order,
    limits = c(0.5, length(celltype_order) + 0.5),
    expand = c(0, 0)
  ) +
  labs(x = "Region", y = "Cell type") +
  theme_classic(base_size = 14) +
  theme(
    axis.text.x = element_text(color = "black", size = 16, face = "bold"),
    axis.text.y = element_text(color = "black", size = 16, face = "bold"),
    axis.title = element_text(face = "bold", size = 16),
    legend.title = element_text(face = "bold", size = 14),
    legend.text = element_text(size = 13),
    legend.position = "right"
  )


In [ ]:
p

In [ ]:
ggsave(file.path(rega_root, "02_run_rega/pathology_association/figures/selected_modules_pathology_heatmap.pdf"),
  p, width = 10.5, height = 5.5
)


In [ ]:
#ggsave(
#  file.path(out_dir, "positive_AD_modules_heatmap.png"),
#  p,
#  width = 9,
#  height = 5.5,
#  dpi = 300
#)